# Lineborn v6 deployment
Reconstruct/verify the trained adapter export from Google Drive, merge the DPO LoRA into Qwen3-4B-Instruct-2507, convert to GGUF, and build Balanced (Q4_K_M) and Performance (Q8_0) candidates. The shipping runtime is not changed by this notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!rm -rf /content/lineborn-runtime
!git clone -b lineborn-v6-deployment https://github.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime.git /content/lineborn-runtime
%cd /content/lineborn-runtime
!python -m pip install -q -r training/requirements-colab.txt


In [ ]:
from pathlib import Path
import shutil
drive_root = Path('/content/drive/MyDrive')
parts = [drive_root / f'lineborn-sales-adapters-v6.zip.part{i}' for i in range(1, 9)]
missing = [str(p) for p in parts if not p.exists()]
assert not missing, f'Missing split files: {missing}'
archive = Path('/content/lineborn-sales-adapters-v6.zip')
with archive.open('wb') as out:
    for p in parts:
        with p.open('rb') as src:
            shutil.copyfileobj(src, out, length=8*1024*1024)
print(archive, archive.stat().st_size)


In [ ]:
!python training/verify_lineborn_v6_export.py /content/lineborn-sales-adapters-v6.zip --extract-dir /content/lineborn-v6-adapters


In [ ]:
!bash training/export_lineborn_v6_gguf.sh /content/lineborn-v6-adapters/lineborn-sales-dpo/adapter /content/lineborn-v6-export


In [ ]:
from pathlib import Path
import shutil
src = Path('/content/lineborn-v6-export')
dst = Path('/content/drive/MyDrive/Lineborn-v6-export')
dst.mkdir(parents=True, exist_ok=True)
for name in ['lineborn-v6-balanced-q4_k_m.gguf','lineborn-v6-performance-q8_0.gguf','Modelfile.balanced','Modelfile.performance','lineborn-v6-deployment-manifest.json']:
    shutil.copy2(src/name, dst/name)
print('Saved candidates to', dst)


## Next gate
Import both GGUFs into Ollama as `lineborn-v6-balanced` and `lineborn-v6-performance`, then run `benchmarks/lineborn_trained_release_acceptance_v6.py --output-dir ...`. Do not switch the beta runtime until both candidates clear the strict gate.
